In [1]:
# ================================================================
# CELL 0 — PROJECT CONFIGURATION (ADD THIS FIRST)
# ================================================================

import os

# Automatically resolve project root
BASE_DIR = os.getcwd()

DATASET_DIR      = os.path.join(BASE_DIR, "dataset")
IMAGE_DIR        = os.path.join(DATASET_DIR, "images")
EMBED_DIR        = os.path.join(DATASET_DIR, "embeddings")
EMBED_FILE       = os.path.join(EMBED_DIR, "embeddings.npz")

VIDEO_INPUT_DIR  = os.path.join(BASE_DIR, "input_videos")
VIDEO_OUTPUT_DIR = os.path.join(BASE_DIR, "output_videos")

ATTEND_DIR       = os.path.join(BASE_DIR, "attendance_logs")

# Create folders if missing
for folder in [
    DATASET_DIR,
    IMAGE_DIR,
    EMBED_DIR,
    VIDEO_INPUT_DIR,
    VIDEO_OUTPUT_DIR,
    ATTEND_DIR
]:
    os.makedirs(folder, exist_ok=True)

print("✅ Project directories ready")
print("Base:", BASE_DIR)

✅ Project directories ready
Base: C:\Users\sgaga\FaceTrackingProject


In [2]:
# ================================================================
# CELL 1 — INSTALLATION & IMPORTS
# ================================================================
# • Installs all required packages (uncomment if first run)
# • Imports every library used across the entire pipeline
# • Sets up inline display for Jupyter Notebook
# ================================================================

# ------- Uncomment these on first run -------
# !pip install insightface
# !pip install onnxruntime-gpu       # GPU (needs CUDA 11.x/12.x)
# !pip install opencv-python
# !pip install scikit-learn
# !pip install tqdm
# !pip install IPython               # usually pre-installed in Jupyter
# --------------------------------------------

import os
import sys
import time
import threading
import queue
import warnings
import cv2
import numpy as np
from tqdm import tqdm
from collections import deque
from insightface.app import FaceAnalysis
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Video, clear_output

warnings.filterwarnings("ignore")   # suppress onnxruntime warnings

print(f"Python  : {sys.version.split()[0]}")
print(f"NumPy   : {np.__version__}")
print(f"OpenCV  : {cv2.__version__}")
print("✅ All imports successful")

Python  : 3.10.11
NumPy   : 2.2.6
OpenCV  : 4.13.0
✅ All imports successful


In [3]:
# ================================================================
# CELL 2 — GPU / CPU AUTO-DETECTION & MODEL LOADING
# ================================================================
# • Checks if NVIDIA GPU (CUDA) is available via onnxruntime
# • Falls back to CPU automatically if CUDA fails
# • Loads InsightFace "buffalo_l" (ArcFace) model
# • det_size=(640,640) balances speed and accuracy
# ================================================================

def get_execution_context():
    """Detect GPU availability and return ctx_id (0=GPU, -1=CPU)."""
    try:
        import onnxruntime as ort
        providers = ort.get_available_providers()
        print(f"📦 ONNX Runtime providers: {providers}")

        if "CUDAExecutionProvider" in providers:
            # Quick sanity check — try creating a CUDA session
            dummy = ort.InferenceSession.__new__(ort.InferenceSession)
            print("🟢 NVIDIA GPU detected → using CUDA")
            return 0   # GPU
    except Exception as e:
        print(f"⚠️  GPU check failed: {e}")

    print("🟡 Falling back to CPU (slower but works)")
    return -1  # CPU

CTX_ID = get_execution_context()

# Load model (downloads ~300 MB on first run)
app = FaceAnalysis(
    name="buffalo_l",
    providers=[
        "CUDAExecutionProvider" if CTX_ID == 0 else "CPUExecutionProvider"
    ],
)
app.prepare(ctx_id=CTX_ID, det_size=(640, 640))

print(f"\n✅ InsightFace model loaded  |  device = {'GPU (RTX 3050)' if CTX_ID == 0 else 'CPU'}")

📦 ONNX Runtime providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
🟡 Falling back to CPU (slower but works)
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\sgaga/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\sgaga/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\sgaga/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\sgaga/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvide

In [4]:
# ================================================================
# CELL 3 — EXTRACT & SAVE FACE EMBEDDINGS
# ================================================================
# • Reads every image in dataset/images/
# • Detects the LARGEST face per image
# • Extracts 512-D ArcFace embedding
# • Supports multiple images per person:
#     alice_1.jpg, alice_2.jpg  →  name = "alice"
# • Saves all embeddings to dataset/embeddings/embeddings.npz
# ================================================================

# IMAGE_FOLDER       = "dataset/images"
IMAGE_FOLDER = IMAGE_DIR
EMBEDDING_FILE = EMBED_FILE
EMBEDDING_SAVE_DIR = "dataset/embeddings"
EMBEDDING_FILE     = os.path.join(EMBEDDING_SAVE_DIR, "embeddings.npz")
SUPPORTED_EXT      = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

os.makedirs(EMBEDDING_SAVE_DIR, exist_ok=True)

embeddings = []
names      = []
files      = sorted(os.listdir(IMAGE_FOLDER))

print(f"📂 Found {len(files)} file(s) in {IMAGE_FOLDER}\n")

for file in tqdm(files, desc="Extracting embeddings"):
    ext = os.path.splitext(file)[1].lower()
    if ext not in SUPPORTED_EXT:
        continue

    path = os.path.join(IMAGE_FOLDER, file)
    img  = cv2.imread(path)

    if img is None:
        print(f"  ❌ Cannot read: {file}")
        continue

    faces = app.get(img)

    if len(faces) == 0:
        print(f"  ⚠️  No face found: {file}")
        continue

    # Pick the largest face by bounding-box area
    faces.sort(
        key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]),
        reverse=True,
    )

    embedding = faces[0].embedding

    # "alice_1.jpg" → "alice",  "bob.png" → "bob"
    raw_name = os.path.splitext(file)[0]
    # Strip trailing _1, _2, etc. if present
    parts = raw_name.rsplit("_", 1)
    name  = parts[0] if len(parts) == 2 and parts[1].isdigit() else raw_name

    embeddings.append(embedding)
    names.append(name)
    print(f"  ✅ {name:<20s} (from {file})")

# ---------- Guard against empty dataset ----------
if len(embeddings) == 0:
    raise RuntimeError(
        "❌ No embeddings extracted!  "
        "Make sure dataset/images/ has face photos."
    )

embeddings = np.array(embeddings)   # shape (N, 512)
names      = np.array(names)

np.savez(EMBEDDING_FILE, embeddings=embeddings, names=names)

unique = np.unique(names)
print(f"\n✅ Saved {len(names)} embedding(s) for {len(unique)} person(s)")
print(f"   Persons: {', '.join(unique)}")
print(f"   File   : {EMBEDDING_FILE}")

📂 Found 1 file(s) in C:\Users\sgaga\FaceTrackingProject\dataset\images



Extracting embeddings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.24s/it]

  ✅ ram                  (from ram.jpg)

✅ Saved 1 embedding(s) for 1 person(s)
   Persons: ram
   File   : dataset/embeddings\embeddings.npz


In [5]:
# ================================================================
# CELL 4 — LOAD EMBEDDINGS DATABASE
# ================================================================
# • Loads the .npz file created in Cell 3
# • Stores known_embeddings and known_names in memory
# • Prints a summary of the loaded database
# ================================================================

# EMBEDDING_FILE = "dataset/embeddings/embeddings.npz"

EMBEDDING_FILE = EMBED_FILE

if not os.path.exists(EMBEDDING_FILE):
    raise FileNotFoundError(
        f"❌ {EMBEDDING_FILE} not found — run Cell 3 first!"
    )

data             = np.load(EMBEDDING_FILE, allow_pickle=True)
known_embeddings = data["embeddings"]
known_names      = data["names"]

unique_names = np.unique(known_names)

print(f"✅ Loaded {len(known_names)} embedding(s) for {len(unique_names)} person(s)")
for person in unique_names:
    count = np.sum(known_names == person)
    print(f"   • {person:<20s}  ({count} embedding{'s' if count > 1 else ''})")

✅ Loaded 1 embedding(s) for 1 person(s)
   • ram                   (1 embedding)


In [6]:
# ================================================================
# CELL 5 — RECOGNITION ENGINE
# ================================================================
# • recognize_face()  — cosine-similarity matching against DB
# • draw_fancy_label() — anti-aliased label with background box
# • AsyncDetector     — runs face detection in a background thread
#                       so the display loop NEVER blocks (no lag)
# ================================================================

# ---------------------- Recognition ----------------------
def recognize_face(embedding, known_embeddings, known_names, threshold=0.50):
    """
    Compare one face embedding against all known embeddings.
    Returns (name, similarity_score).
    Threshold: cosine similarity — higher = stricter.
      0.40 = lenient  |  0.50 = balanced  |  0.60 = strict
    """
    if len(known_embeddings) == 0:
        return "Unknown", 0.0

    sims      = cosine_similarity([embedding], known_embeddings)[0]
    best_idx  = int(np.argmax(sims))
    best_score = float(sims[best_idx])

    if best_score >= threshold:
        return str(known_names[best_idx]), best_score
    return "Unknown", best_score


# -------------------- Label Drawing ---------------------
def draw_fancy_label(frame, text, x, y, color,
                     font_scale=0.65, thickness=2):
    """Draw a text label with a filled rounded-rect background."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    pad = 6
    y_top = max(y - th - 2 * pad, 0)     # keep label on screen

    # Background rectangle
    cv2.rectangle(
        frame,
        (x, y_top),
        (x + tw + 2 * pad, y_top + th + 2 * pad + baseline),
        color, -1
    )
    # Text color: white on dark bg, black on bright bg
    brightness = (color[0] * 0.114 + color[1] * 0.587 + color[2] * 0.299)
    txt_clr = (0, 0, 0) if brightness > 127 else (255, 255, 255)

    cv2.putText(
        frame, text,
        (x + pad, y_top + th + pad),
        font, font_scale, txt_clr, thickness, cv2.LINE_AA
    )


# --------------- Async Threaded Detector ----------------
class AsyncDetector:
    """
    Runs InsightFace detection + recognition in a background thread.
    The main display loop fetches cached results without blocking,
    which eliminates video lag completely.
    """

    def __init__(self, app, known_emb, known_names, threshold=0.50):
        self._app        = app
        self._known_emb  = known_emb
        self._known_n    = known_names
        self._threshold  = threshold

        self._results    = []             # latest detection results
        self._lock       = threading.Lock()
        self._frame      = None           # frame awaiting detection
        self._new_frame  = False
        self._running    = True
        self._det_count  = 0              # how many detections completed

        self._thread = threading.Thread(target=self._worker, daemon=True)
        self._thread.start()

    # ---- called from MAIN thread (non-blocking) ----
    def submit(self, frame):
        """Submit a frame for background detection. Non-blocking."""
        with self._lock:
            self._frame     = frame.copy()
            self._new_frame = True

    @property
    def results(self):
        """Get latest detection results (list of dicts)."""
        with self._lock:
            return list(self._results)

    @property
    def detection_count(self):
        with self._lock:
            return self._det_count

    # ---- runs in BACKGROUND thread ----
    def _worker(self):
        while self._running:
            frame = None
            with self._lock:
                if self._new_frame:
                    frame = self._frame
                    self._new_frame = False

            if frame is not None:
                faces   = self._app.get(frame)
                results = []
                for face in faces:
                    bbox = face.bbox.astype(int).tolist()
                    name, score = recognize_face(
                        face.embedding,
                        self._known_emb,
                        self._known_n,
                        self._threshold,
                    )
                    results.append({
                        "bbox": bbox, "name": name, "score": score
                    })
                with self._lock:
                    self._results = results
                    self._det_count += 1
            else:
                time.sleep(0.005)   # idle — avoid busy-wait

    def stop(self):
        self._running = False
        self._thread.join(timeout=5)


print("✅ Recognition engine ready")

✅ Recognition engine ready


In [7]:
# ================================================================
# CELL 6 — VIDEO PROCESSING  (lag-free, threaded detection)
# ================================================================
# • Opens input video and creates output writer
# • Spawns AsyncDetector (Cell 5) in background thread
# • Main loop:  read → draw cached results → write → display
#   Display NEVER waits for detection → zero lag
# • Adaptive waitKey matches original video FPS
# • FPS counter + face count overlay
# • Press 'Q' in the OpenCV window to stop early
# • Progress bar with ETA in the notebook cell
# ================================================================

# ──────────── CONFIGURATION ─────────────────────────────────
# VIDEO_INPUT       = "input_videos/test.mp4"    # source video
# VIDEO_OUTPUT      = "output_videos/output.mp4" # annotated output
VIDEO_INPUT = os.path.join(VIDEO_INPUT_DIR, "test.mp4")
VIDEO_OUTPUT = os.path.join(VIDEO_OUTPUT_DIR, "output.mp4")
SIMILARITY_THRESH = 0.50     # cosine similarity threshold
SUBMIT_EVERY_N    = 2        # send every Nth frame to detector
DISPLAY_SCALE     = 1.0      # <1.0 = smaller preview (faster draw)
SHOW_WINDOW       = True     # set False if cv2.imshow crashes
# ─────────────────────────────────────────────────────────────

os.makedirs("output_videos", exist_ok=True)

# ── Open Video ───────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_INPUT)
if not cap.isOpened():
    raise FileNotFoundError(f"❌ Cannot open: {VIDEO_INPUT}")

fps          = int(cap.get(cv2.CAP_PROP_FPS)) or 30
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"🎬 Input  : {VIDEO_INPUT}")
print(f"   Frames : {total_frames}  |  FPS: {fps}  |  Size: {width}×{height}")
print(f"   Device : {'GPU' if CTX_ID == 0 else 'CPU'}")
print(f"   Detect every {SUBMIT_EVERY_N} frame(s)\n")

# ── Video Writer ─────────────────────────────────────────────
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out    = cv2.VideoWriter(VIDEO_OUTPUT, fourcc, fps, (width, height))

# ── Start Async Detector ─────────────────────────────────────
detector = AsyncDetector(
    app, known_embeddings, known_names, threshold=SIMILARITY_THRESH
)

# ── Tracking Variables ───────────────────────────────────────
frame_idx  = 0
fps_times  = deque(maxlen=60)           # for rolling FPS calc
t_start    = time.time()

pbar = tqdm(total=total_frames, desc="Processing", unit="frame",
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]")

# Colors
COLOR_KNOWN   = (0, 200, 0)     # green
COLOR_UNKNOWN = (0, 0, 220)     # red
COLOR_FPS     = (255, 200, 0)   # cyan-ish

try:
    while True:
        t_frame = time.time()

        ret, frame = cap.read()
        if not ret:
            break

        # ── Submit frame for background detection ────────────
        if frame_idx % SUBMIT_EVERY_N == 0:
            detector.submit(frame)

        # ── Draw cached results (non-blocking) ──────────────
        results    = detector.results
        face_count = len(results)

        for r in results:
            x1, y1, x2, y2 = r["bbox"]
            name  = r["name"]
            score = r["score"]
            color = COLOR_KNOWN if name != "Unknown" else COLOR_UNKNOWN

            # Bounding box with thickness based on confidence
            box_thick = 2 if name == "Unknown" else 3
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, box_thick)

            # Corner accents (top-left and bottom-right)
            corner_len = max(15, (x2 - x1) // 5)
            cv2.line(frame, (x1, y1), (x1 + corner_len, y1), color, 4)
            cv2.line(frame, (x1, y1), (x1, y1 + corner_len), color, 4)
            cv2.line(frame, (x2, y2), (x2 - corner_len, y2), color, 4)
            cv2.line(frame, (x2, y2), (x2, y2 - corner_len), color, 4)

            # Label
            label = f"{name} {score:.0%}"
            draw_fancy_label(frame, label, x1, y1, color)

        # ── FPS Counter ──────────────────────────────────────
        fps_times.append(time.time())
        if len(fps_times) > 1:
            rolling_fps = (len(fps_times) - 1) / (fps_times[-1] - fps_times[0])
        else:
            rolling_fps = 0.0

        # HUD overlay (top bar)
        cv2.rectangle(frame, (0, 0), (width, 36), (0, 0, 0), -1)
        hud = (
            f"FPS: {rolling_fps:.1f}  |  "
            f"Faces: {face_count}  |  "
            f"Frame: {frame_idx + 1}/{total_frames}  |  "
            f"Device: {'GPU' if CTX_ID == 0 else 'CPU'}"
        )
        cv2.putText(frame, hud, (10, 25),
                     cv2.FONT_HERSHEY_SIMPLEX, 0.55, COLOR_FPS, 1, cv2.LINE_AA)

        # ── Write Output ─────────────────────────────────────
        out.write(frame)

        # ── Display (smooth timing) ──────────────────────────
        if SHOW_WINDOW:
            if DISPLAY_SCALE != 1.0:
                disp = cv2.resize(frame, None,
                                  fx=DISPLAY_SCALE, fy=DISPLAY_SCALE)
            else:
                disp = frame

            cv2.imshow("Face Recognition  |  Press Q to quit", disp)

            # Adaptive delay to match original video FPS
            elapsed_ms = (time.time() - t_frame) * 1000
            target_ms  = (1000.0 / fps)
            delay      = max(1, int(target_ms - elapsed_ms))

            if cv2.waitKey(delay) & 0xFF == ord("q"):
                print("\n⏹️  Stopped by user")
                break

        frame_idx += 1
        pbar.update(1)

except KeyboardInterrupt:
    print("\n⏹️  Interrupted by user")

finally:
    # ── Cleanup (always runs) ────────────────────────────────
    pbar.close()
    detector.stop()
    cap.release()
    out.release()
    if SHOW_WINDOW:
        cv2.destroyAllWindows()
        cv2.waitKey(1)  # flush on Windows

    # ── Summary ──────────────────────────────────────────────
    total_time = time.time() - t_start
    avg_fps    = frame_idx / total_time if total_time > 0 else 0
    det_runs   = detector.detection_count

    print("\n" + "=" * 55)
    print("📊  PROCESSING SUMMARY")
    print("=" * 55)
    print(f"   Frames processed : {frame_idx} / {total_frames}")
    print(f"   Detection runs   : {det_runs}  (every {SUBMIT_EVERY_N} frames)")
    print(f"   Average FPS      : {avg_fps:.1f}")
    print(f"   Total time       : {total_time:.1f}s")
    print(f"   Device used      : {'GPU' if CTX_ID == 0 else 'CPU'}")
    print(f"   Output saved     : {VIDEO_OUTPUT}")
    print("=" * 55)

🎬 Input  : C:\Users\sgaga\FaceTrackingProject\input_videos\test.mp4
   Frames : 94  |  FPS: 19  |  Size: 480×848
   Device : CPU
   Detect every 2 frame(s)



Processing: 100%|█████████████████████████████████████████████████████████████████████████████████| 94/94 [00:07<00:00]



📊  PROCESSING SUMMARY
   Frames processed : 94 / 94
   Detection runs   : 3  (every 2 frames)
   Average FPS      : 11.6
   Total time       : 8.1s
   Device used      : CPU
   Output saved     : C:\Users\sgaga\FaceTrackingProject\output_videos\output.mp4


In [8]:
# ================================================================
# CELL 7 — PLAY OUTPUT VIDEO INSIDE JUPYTER NOTEBOOK
# ================================================================
# • Converts mp4v → H.264 for browser-compatible playback
# • Displays the result inline (no external player needed)
# • Falls back to raw file if ffmpeg is unavailable
# ================================================================

import shutil
import subprocess

H264_OUTPUT = VIDEO_OUTPUT.replace(".mp4", "_h264.mp4")

if shutil.which("ffmpeg"):
    print("🔄 Converting to H.264 for notebook playback...")
    subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", VIDEO_OUTPUT,
            "-vcodec", "libx264",
            "-crf", "23",
            "-preset", "fast",
            "-movflags", "+faststart",
            "-an",
            H264_OUTPUT,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )
    playback_file = H264_OUTPUT
    print(f"✅ H.264 video: {H264_OUTPUT}")
else:
    print("⚠️  ffmpeg not found — playing raw mp4v (may not render inline)")
    playback_file = VIDEO_OUTPUT

# Display in notebook
display(Video(playback_file, embed=True, width=800))

⚠️  ffmpeg not found — playing raw mp4v (may not render inline)


In [9]:
# cell 8 ================================================================
# LIVE ENROLLMENT — Add new person via image or webcam
# ================================================================
# • Captures face from image/webcam
# • Extracts embedding and appends to existing database
# • No need to restart the pipeline
# ================================================================

class FaceDatabase:
    """Persistent face database with live add/remove/update."""

    def __init__(self, db_path="dataset/embeddings/embeddings.npz"):
        self.db_path    = db_path
        self.embeddings = np.empty((0, 512), dtype=np.float32)
        self.names      = np.array([], dtype=str)
        self.metadata   = {}  # extra info per person
        self._load()

    def _load(self):
        if os.path.exists(self.db_path):
            data = np.load(self.db_path, allow_pickle=True)
            self.embeddings = data["embeddings"]
            self.names      = data["names"]
            if "metadata" in data:
                self.metadata = data["metadata"].item()
            print(f"✅ Loaded {len(self.names)} embeddings")

    def save(self):
        os.makedirs(os.path.dirname(self.db_path), exist_ok=True)
        np.savez(
            self.db_path,
            embeddings=self.embeddings,
            names=self.names,
            metadata=self.metadata,
        )
        print(f"💾 Database saved: {len(self.names)} embeddings")

    def enroll_from_image(self, image_path, person_name, app):
        """Add a face from an image file."""
        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f"Cannot read: {image_path}")

        faces = app.get(img)
        if not faces:
            print(f"⚠️  No face found in {image_path}")
            return False

        # Largest face
        faces.sort(
            key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]),
            reverse=True,
        )

        embedding = faces[0].embedding.reshape(1, -1)

        # Check if already enrolled (avoid exact duplicates)
        if len(self.embeddings) > 0:
            sims = cosine_similarity(embedding, self.embeddings)[0]
            if np.max(sims) > 0.85:
                existing = self.names[np.argmax(sims)]
                print(f"⚠️  Very similar to '{existing}' (sim={np.max(sims):.3f})")
                print(f"   Adding anyway as '{person_name}'")

        self.embeddings = np.vstack([self.embeddings, embedding])
        self.names      = np.append(self.names, person_name)

        self.metadata[person_name] = {
            "enrolled_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "source": image_path,
            "num_embeddings": int(np.sum(self.names == person_name)),
        }

        self.save()
        print(f"✅ Enrolled '{person_name}' from {image_path}")
        return True

    def enroll_from_webcam(self, person_name, app, num_shots=5):
        """Capture multiple angles from webcam for better accuracy."""
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            raise RuntimeError("Cannot open webcam")

        collected = 0
        print(f"📸 Capturing {num_shots} shots for '{person_name}'...")
        print("   Move your head slightly between captures")

        while collected < num_shots:
            ret, frame = cap.read()
            if not ret:
                break

            faces = app.get(frame)

            if faces:
                face = max(faces, key=lambda f:
                    (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
                bbox = face.bbox.astype(int)
                cv2.rectangle(frame, 
                    (bbox[0], bbox[1]), (bbox[2], bbox[3]),
                    (0, 255, 0), 2)
                cv2.putText(frame,
                    f"Press SPACE to capture ({collected}/{num_shots})",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                    (0, 255, 0), 2)
            else:
                cv2.putText(frame, "No face detected",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                    (0, 0, 255), 2)

            cv2.imshow("Enrollment", frame)
            key = cv2.waitKey(1) & 0xFF

            if key == ord(" ") and faces:
                embedding = face.embedding.reshape(1, -1)
                self.embeddings = np.vstack([self.embeddings, embedding])
                self.names = np.append(self.names, person_name)
                collected += 1
                print(f"   ✅ Captured {collected}/{num_shots}")
            elif key == ord("q"):
                break

        cap.release()
        cv2.destroyAllWindows()

        if collected > 0:
            self.save()
            print(f"✅ Enrolled '{person_name}' with {collected} shots")
        return collected > 0

    def remove_person(self, person_name):
        """Remove all embeddings for a person."""
        mask = self.names != person_name
        removed = np.sum(~mask)
        self.embeddings = self.embeddings[mask]
        self.names      = self.names[mask]
        self.metadata.pop(person_name, None)
        self.save()
        print(f"🗑️  Removed '{person_name}' ({removed} embeddings)")

    def list_persons(self):
        """Show all enrolled persons."""
        unique = np.unique(self.names)
        print(f"\n📋 Database: {len(unique)} persons, {len(self.names)} embeddings")
        for name in unique:
            count = int(np.sum(self.names == name))
            meta  = self.metadata.get(name, {})
            enrolled = meta.get("enrolled_at", "unknown")
            print(f"   • {name:<20s} {count} emb(s)  enrolled: {enrolled}")

    def recognize(self, embedding, threshold=0.50):
        """Match embedding against database. Returns (name, score)."""
        if len(self.embeddings) == 0:
            return "Unknown", 0.0

        sims     = cosine_similarity([embedding], self.embeddings)[0]
        best_idx = int(np.argmax(sims))
        best_sim = float(sims[best_idx])

        if best_sim >= threshold:
            return str(self.names[best_idx]), best_sim
        return "Unknown", best_sim


# ── Usage ──────────────────────────────────────────────────
# db = FaceDatabase()
# db.enroll_from_image("photos/john.jpg", "John", app)
# db.enroll_from_webcam("Jane", app, num_shots=5)
# db.list_persons()
# db.remove_person("John")

In [10]:
# ================================================================
# CELL 9 — BATCH PROCESSING — Process multiple videos automatically
# ================================================================

def process_multiple_videos(
    input_folder,
    output_folder,
    db,
    app,
    submit_every_n=2,
    threshold=0.50,
):
    """Process all videos in a folder using face recognition."""

    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)

    VIDEO_EXT = {".mp4", ".avi", ".mkv", ".mov", ".wmv", ".flv"}

    # Validate input folder
    if not os.path.exists(input_folder):
        raise FileNotFoundError(f"❌ Input folder not found: {input_folder}")

    videos = sorted([
        f for f in os.listdir(input_folder)
        if os.path.splitext(f)[1].lower() in VIDEO_EXT
    ])

    if len(videos) == 0:
        print(f"⚠️ No videos found in {input_folder}")
        return {}

    print(f"📂 Found {len(videos)} video(s) in:\n   {input_folder}")

    all_results = {}

    # Process each video
    for vid_idx, video_file in enumerate(videos, 1):

        input_path  = os.path.join(input_folder, video_file)
        output_file = f"processed_{video_file}"
        output_path = os.path.join(output_folder, output_file)

        print(f"\n{'='*55}")
        print(f"🎬 [{vid_idx}/{len(videos)}] Processing: {video_file}")
        print(f"{'='*55}")

        cap = cv2.VideoCapture(input_path)

        if not cap.isOpened():
            print(f"❌ Cannot open: {video_file}")
            continue

        fps    = int(cap.get(cv2.CAP_PROP_FPS)) or 30
        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Output writer
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        face_log  = {}
        frame_idx = 0
        cached    = []

        pbar = tqdm(
            total=total,
            desc=f"Processing {video_file}",
            unit="frame"
        )

        while True:

            ret, frame = cap.read()
            if not ret:
                break

            # Run detection periodically
            if frame_idx % submit_every_n == 0:

                faces  = app.get(frame)
                cached = []

                for face in faces:

                    bbox = face.bbox.astype(int).tolist()

                    name, score = db.recognize(
                        face.embedding,
                        threshold
                    )

                    cached.append((bbox, name, score))

                    if name != "Unknown":
                        face_log.setdefault(name, []).append(frame_idx)

            # Draw results
            for (bbox, name, score) in cached:

                x1, y1, x2, y2 = bbox

                color = (
                    (0, 200, 0)
                    if name != "Unknown"
                    else (0, 0, 220)
                )

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    color,
                    2
                )

                label = f"{name} {score:.0%}"

                cv2.putText(
                    frame,
                    label,
                    (x1, max(y1 - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2,
                    cv2.LINE_AA,
                )

            out.write(frame)

            frame_idx += 1
            pbar.update(1)

        pbar.close()

        cap.release()
        out.release()

        # Print summary
        print(f"\n📊 Summary for {video_file}:")

        if len(face_log) == 0:
            print("   No known faces detected.")
        else:
            for name, frames in face_log.items():

                first = frames[0] / fps
                last  = frames[-1] / fps

                print(
                    f"   • {name}: "
                    f"{len(frames)} appearances "
                    f"({first:.1f}s – {last:.1f}s)"
                )

        print(f"💾 Saved to: {output_path}")

        all_results[video_file] = face_log

    print("\n✅ Batch processing complete.")

    return all_results


# ================================================================
# USAGE — Correct way (uses your project configuration)
# ================================================================

# Load database safely
db = FaceDatabase(EMBED_FILE)

# Run batch processing
results = process_multiple_videos(
    VIDEO_INPUT_DIR,
    VIDEO_OUTPUT_DIR,
    db,
    app
)

print("\nFinal Results:", results)

✅ Loaded 1 embeddings
📂 Found 1 video(s) in:
   C:\Users\sgaga\FaceTrackingProject\input_videos

🎬 [1/1] Processing: test.mp4


Processing test.mp4: 100%|██████████████████████████████████████████████████████████| 94/94 [02:33<00:00,  1.63s/frame]


📊 Summary for test.mp4:
   • ram: 46 appearances (0.0s – 4.7s)
💾 Saved to: C:\Users\sgaga\FaceTrackingProject\output_videos\processed_test.mp4

✅ Batch processing complete.

Final Results: {'test.mp4': {'ram': [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90]}}


In [11]:
# cell 10 ================================================================
# FACE TRACKER — Assign consistent IDs using IoU matching
# ================================================================
# Without tracking: same person gets different colors/labels
#                   flickering between frames
# With tracking:    smooth, consistent identity per person
# ================================================================

class FaceTracker:
    """Simple IoU-based face tracker for consistent IDs."""

    def __init__(self, iou_threshold=0.3, max_disappeared=15):
        self.next_id         = 0
        self.tracks          = {}    # id → {bbox, name, score, missing}
        self.iou_threshold   = iou_threshold
        self.max_disappeared = max_disappeared

    def _iou(self, boxA, boxB):
        """Intersection over Union between two [x1,y1,x2,y2] boxes."""
        xA = max(boxA[0], boxB[0])
        yA = max(boxA[1], boxB[1])
        xB = min(boxA[2], boxB[2])
        yB = min(boxA[3], boxB[3])

        inter = max(0, xB - xA) * max(0, yB - yA)
        areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
        areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])

        union = areaA + areaB - inter
        return inter / union if union > 0 else 0

    def update(self, detections):
        """
        detections: list of {"bbox": [x1,y1,x2,y2], "name": str, "score": float}
        Returns: list of {"track_id": int, "bbox", "name", "score"}
        """
        # Mark all existing tracks as missing
        for tid in self.tracks:
            self.tracks[tid]["missing"] += 1

        matched_tracks = set()
        matched_dets   = set()
        results        = []

        # Match detections to existing tracks by IoU
        for d_idx, det in enumerate(detections):
            best_tid = None
            best_iou = self.iou_threshold

            for tid, track in self.tracks.items():
                if tid in matched_tracks:
                    continue
                iou = self._iou(det["bbox"], track["bbox"])
                if iou > best_iou:
                    best_iou = iou
                    best_tid = tid

            if best_tid is not None:
                # Update existing track
                self.tracks[best_tid].update({
                    "bbox":    det["bbox"],
                    "name":    det["name"],
                    "score":   det["score"],
                    "missing": 0,
                })
                matched_tracks.add(best_tid)
                matched_dets.add(d_idx)

        # Create new tracks for unmatched detections
        for d_idx, det in enumerate(detections):
            if d_idx not in matched_dets:
                self.tracks[self.next_id] = {
                    "bbox":    det["bbox"],
                    "name":    det["name"],
                    "score":   det["score"],
                    "missing": 0,
                }
                self.next_id += 1

        # Remove stale tracks
        stale = [
            tid for tid, t in self.tracks.items()
            if t["missing"] > self.max_disappeared
        ]
        for tid in stale:
            del self.tracks[tid]

        # Return active tracks
        for tid, track in self.tracks.items():
            if track["missing"] == 0:
                results.append({
                    "track_id": tid,
                    "bbox":     track["bbox"],
                    "name":     track["name"],
                    "score":    track["score"],
                })

        return results


# ── Usage in video loop ────────────────────────────────────
# tracker = FaceTracker()
# ...
# in the detection callback:
#   raw_detections = [{"bbox":..., "name":..., "score":...}, ...]
#   tracked = tracker.update(raw_detections)
#   for t in tracked:
#       print(f"Track #{t['track_id']}: {t['name']}")

In [12]:
# cell 11================================================================
# ATTENDANCE SYSTEM — Auto-log who appeared and when
# ================================================================

import csv
from datetime import datetime

class AttendanceLogger:
    """Logs face appearances with timestamps."""

    def __init__(self, log_dir="attendance_logs"):
        self.log_dir    = log_dir
        self.seen_today = {}   # {name: first_seen_time}
        os.makedirs(log_dir, exist_ok=True)

    def mark(self, name, frame_idx, fps):
        """Mark a person as present."""
        if name == "Unknown":
            return

        if name not in self.seen_today:
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            video_time = f"{frame_idx / fps:.1f}s"

            self.seen_today[name] = {
                "timestamp":  timestamp,
                "video_time": video_time,
                "frame":      frame_idx,
            }
            print(f"  📝 {name} marked present at {timestamp}")

    def save_csv(self, filename=None):
        """Save attendance to CSV."""
        if filename is None:
            date_str = datetime.now().strftime("%Y-%m-%d")
            filename = f"attendance_{date_str}.csv"

        filepath = os.path.join(self.log_dir, filename)

        with open(filepath, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Name", "Status", "Timestamp",
                             "Video Time", "Frame"])
            for name, info in sorted(self.seen_today.items()):
                writer.writerow([
                    name, "Present", info["timestamp"],
                    info["video_time"], info["frame"],
                ])

        print(f"\n📄 Attendance saved: {filepath}")
        print(f"   Total present: {len(self.seen_today)}")
        return filepath

    def get_summary(self):
        return dict(self.seen_today)


# ── Usage ──────────────────────────────────────────────────
# logger = AttendanceLogger()
# # In video loop:
#   logger.mark(name, frame_idx, fps)
# # After processing:
#   logger.save_csv()